# Infra-FM: Prithvi-EO-2.0-300M-TL Linear Probe

Self-contained Prithvi-EO-2.0 evaluation notebook, parallel to
`infra_fm_satlas_s1_eval.ipynb` and `infra_fm_satlas_multisector_v1.ipynb`.
**Does not modify those.**

Model: `ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL`
- 300M-parameter ViT with 3D patch embedding (B, C, T, H, W).
- TL = Temporal/Location embeddings (passed as `None` for our single-timestamp
  data; the model handles missing temporal/location metadata gracefully).
- Optical-only — no SAR variant exists, so only one Prithvi notebook is needed.

Band mapping:
- Prithvi expects 6 bands in order **Blue, Green, Red, Narrow NIR (B8A),
  SWIR1, SWIR2** = S2 bands **B02, B03, B04, B8A, B11, B12**.
- Our 9-band `.npy` storage order is `[B04, B03, B02, B08, B8A, B11, B12, VV, VH]`.
  Indices to pull, in Prithvi's expected order: **`[2, 1, 0, 4, 5, 6]`**
  (B02=idx2, B03=idx1, B04=idx0, B8A=idx4, B11=idx5, B12=idx6).

> **Spec correction.** The task spec said `[0, 1, 2, 4, 5, 6]` "skipping
> WIDE NIR B08 at index 3" — that assumed `.npy` storage is `[B02, B03,
> B04, ...]`. Our actual storage is `[B04, B03, B02, B08, B8A, B11, B12, VV, VH]`
> (per `curation/stac_imagery.py`: B04 first, then B03, then B02). The
> correct indices are `[2, 1, 0, 4, 5, 6]`. I flagged this rather than
> follow the spec verbatim, because using `[0, 1, 2, 4, 5, 6]` would feed
> Prithvi B04 as Blue, B03 as Green, B02 as Red — a wavelength-channel
> swap that the model was never trained on.

Linear probe protocol (matches the SatlasPretrain S1 run for direct
comparability):
- 25 epochs, batch 16, AdamW lr=1e-3, class weights capped at 10×.
- 7-region multi-sector v1 (13 classes), stratified 0.7/0.15/0.15 split.
- Frozen backbone; classifier head only.
- **Best-val checkpoint is restored before the final test evaluation.**
  This was a methodological gap in the earlier S1 run (test ran on
  final-epoch state). Built in from the start here — search this notebook
  for `BEST_CKPT_BEFORE_TEST` to find the load site.

The smoke check (cell 10) is gated by `SMOKE_ONLY=True` by default. Run
cells 1-10 to verify the model loads and a forward + training step works,
then flip `SMOKE_ONLY=False` and re-run cells 10 and 11 to start training.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Runtime -> Change runtime type -> GPU before training.')


Mounted at /content/drive
GPU available: True
GPU: Tesla T4
Memory: 15.6 GB


In [2]:
# Sanity check: does sklearn import cleanly on fresh runtime?
import sklearn
from sklearn.metrics import f1_score, confusion_matrix
print(f'sklearn {sklearn.__version__} works')

import numpy as np
print(f'numpy {np.__version__}')

import scipy
print(f'scipy {scipy.__version__}')

sklearn 1.6.1 works
numpy 2.0.2
scipy 1.16.3


In [3]:
%%capture
# terratorch pulls timm + huggingface_hub + transformers, but we install
# explicitly so the fallback path works even if terratorch import fails.
!pip install -q terratorch transformers huggingface_hub scikit-learn

# Quick sanity check (printed; %%capture suppresses the pip output but lets
# the python below through if we drop the magic later).


In [4]:
# Undo my bad pin — restore numpy 2.x
!pip install --upgrade --force-reinstall "numpy>=2.2"

# Reinstall sklearn against the current numpy
!pip install --upgrade --force-reinstall --no-deps scikit-learn scipy

  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 1.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 118.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 21.8 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [5]:
# Prithvi-EO-2.0-300M-TL is a *public* model on HuggingFace, but logging in
# raises rate limits and avoids occasional 401s on the first weight download.
# Put your token in Colab Secrets as HF_TOKEN (sidebar -> key icon -> add
# secret). The cell is robust to a missing token — it just logs a warning.
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in to HuggingFace.')
else:
    print('No HF_TOKEN in Colab Secrets — proceeding anonymously. '
          'If model download 401s, add HF_TOKEN as a Colab Secret.')


No HF_TOKEN in Colab Secrets — proceeding anonymously. If model download 401s, add HF_TOKEN as a Colab Secret.


In [6]:
import os, sys, zipfile
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
CODE_ZIP   = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
EXTRACT_TO = '/content/infrabench_repo'

if not Path(f'{EXTRACT_TO}/downstream').exists() and not Path(f'{EXTRACT_TO}/infra_fm_code_only/downstream').exists():
    if not Path(CODE_ZIP).exists():
        print(f'WARNING: no code zip at {CODE_ZIP}. The notebook is self-contained '
              f'and does not need this, but if you want repo helpers available, '
              f'upload infra_fm_curation.zip to {DRIVE_ROOT}/code/.')
    else:
        print(f'Extracting {CODE_ZIP} -> {EXTRACT_TO} ...')
        os.makedirs(EXTRACT_TO, exist_ok=True)
        with zipfile.ZipFile(CODE_ZIP, 'r') as z:
            for member in z.namelist():
                clean = member.replace('\\', '/')
                target = os.path.join(EXTRACT_TO, clean)
                if clean.endswith('/'):
                    os.makedirs(target, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(target), exist_ok=True)
                    with z.open(member) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
        print('done.')
else:
    print(f'Code already extracted at {EXTRACT_TO}.')

# Add to sys.path so any future "from downstream.X import Y" works. The
# notebook does NOT depend on repo code for correctness — the PrithviDataset
# class below is self-contained — but adding to path is harmless.
for candidate in [EXTRACT_TO, f'{EXTRACT_TO}/infra_fm_code_only']:
    if Path(candidate).exists() and candidate not in sys.path:
        sys.path.insert(0, candidate)
print(f'sys.path[:2]: {sys.path[:2]}')


Extracting /content/drive/MyDrive/infra_fm/code/infra_fm_curation.zip -> /content/infrabench_repo ...
done.
sys.path[:2]: ['/content/infrabench_repo/infra_fm_code_only', '/content/infrabench_repo']


In [7]:
import os
from pathlib import Path

# ---- Paths ----------------------------------------------------------------
DATASETS_DRIVE = f'{DRIVE_ROOT}/datasets'
DATASETS_LOCAL = '/content/datasets'
OUTPUT_DIR     = f'{DRIVE_ROOT}/results/fm_eval_prithvi_v1'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATASETS_LOCAL, exist_ok=True)

# ---- Coverage -------------------------------------------------------------
REGIONS = [
    'africa', 'asia', 'australia-oceania', 'central-america',
    'europe', 'north-america', 'south-america',
]
SECTORS = ['energy', 'water', 'transport', 'telecom']

# ---- 13-class taxonomy (locked, must match the SatlasPretrain notebooks) ----
CLASS_NAMES = [
    'energy.transmission.substation',
    'energy.distribution.substation',
    'energy.distribution.other',          # pools substation_untyped + substation_minor
    'energy.generation.power_plant',
    'energy.generation.solar_farm',
    'energy.generation.wind_farm',
    'water.wastewater.plant',
    'water.treatment.plant',
    'water.storage_tank',
    'transport.airport',
    'transport.train_station',
    'transport.port_terminal',
    'telecom.data_center',
]
CLASS_TO_IDX        = {n: i for i, n in enumerate(CLASS_NAMES)}
CLASS_IDX_TO_SECTOR = {i: n.split('.')[0] for i, n in enumerate(CLASS_NAMES)}

ASSET_TYPE_MAP = {
    'energy.transmission.substation':         'energy.transmission.substation',
    'energy.distribution.substation':         'energy.distribution.substation',
    'energy.distribution.substation_untyped': 'energy.distribution.other',
    'energy.distribution.substation_minor':   'energy.distribution.other',
    'energy.generation.power_plant':          'energy.generation.power_plant',
    'energy.generation.solar_farm':           'energy.generation.solar_farm',
    'energy.generation.wind_farm':            'energy.generation.wind_farm',
    'water.wastewater.plant':                 'water.wastewater.plant',
    'water.treatment.plant':                  'water.treatment.plant',
    'water.storage_tank':                     'water.storage_tank',
    'transport.airport':                      'transport.airport',
    'transport.train_station':                'transport.train_station',
    'transport.port_terminal':                'transport.port_terminal',
    'telecom.data_center':                    'telecom.data_center',
}

# ---- Prithvi-specific config ----------------------------------------------
# Our 9-band .npy storage order is:
#   [B04, B03, B02, B08, B8A, B11, B12, VV, VH]   indices 0..8
# Prithvi expects, in this exact channel order:
#   [Blue=B02, Green=B03, Red=B04, NarrowNIR=B8A, SWIR1=B11, SWIR2=B12]
# So we pull indices (B02=2, B03=1, B04=0, B8A=4, B11=5, B12=6):
PRITHVI_BAND_INDICES = [2, 1, 0, 4, 5, 6]   # [Blue, Green, Red, NarrowNIR, SWIR1, SWIR2]

# Reflectance normalization. Prithvi-EO-2.0 was pretrained on HLS Sentinel-2/
# Landsat surface reflectance in roughly [0, 1] dynamic range. Our tiles are
# S2 L2A reflectance — same atmospheric correction family but not identical
# (HLS adds BRDF normalization, masking, and harmonization steps S2 L2A
# alone does not). We approximate with the same percentile_normalize used
# in the SatlasPretrain S2 path. The HLS-vs-S2L2A distribution shift is the
# main known transfer-cost risk; we evaluate empirically.
PERC_LO, PERC_HI = 2.0, 98.0

# ---- Training config ------------------------------------------------------
IMAGE_SIZE = 224                # Prithvi-EO-2.0 native input; tiles upsampled from ~61 to 224
SEED       = 42
LP_EPOCHS  = 25                 # mirrors SatlasPretrain S1 capped run
LP_BATCH   = 16
LP_LR      = 1e-3

# ---- Determinism ----------------------------------------------------------
def set_seed(seed: int) -> None:
    import random
    import numpy as np
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

print(f'Regions ({len(REGIONS)}): {REGIONS}')
print(f'Sectors:                  {SECTORS}')
print(f'Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}')
print(f'Prithvi band indices:     {PRITHVI_BAND_INDICES}  (B02, B03, B04, B8A, B11, B12)')
print(f'Reflectance normalization: percentile_normalize [{PERC_LO}, {PERC_HI}] -> [0, 1]')
print(f'Image size:               {IMAGE_SIZE}x{IMAGE_SIZE}')
print(f'Linear probe:             {LP_EPOCHS} epochs, batch {LP_BATCH}, lr {LP_LR}')
print(f'Output dir:               {OUTPUT_DIR}')


Regions (7): ['africa', 'asia', 'australia-oceania', 'central-america', 'europe', 'north-america', 'south-america']
Sectors:                  ['energy', 'water', 'transport', 'telecom']
Classes (13): ['energy.transmission.substation', 'energy.distribution.substation', 'energy.distribution.other', 'energy.generation.power_plant', 'energy.generation.solar_farm', 'energy.generation.wind_farm', 'water.wastewater.plant', 'water.treatment.plant', 'water.storage_tank', 'transport.airport', 'transport.train_station', 'transport.port_terminal', 'telecom.data_center']
Prithvi band indices:     [2, 1, 0, 4, 5, 6]  (B02, B03, B04, B8A, B11, B12)
Reflectance normalization: percentile_normalize [2.0, 98.0] -> [0, 1]
Image size:               224x224
Linear probe:             25 epochs, batch 16, lr 0.001
Output dir:               /content/drive/MyDrive/infra_fm/results/fm_eval_prithvi_v1


In [8]:
import zipfile, shutil, time, re
from pathlib import Path

drive_path = Path(DATASETS_DRIVE)
print(f'Contents of {DATASETS_DRIVE}:')
if drive_path.exists():
    for entry in sorted(drive_path.iterdir()):
        if entry.suffix == '.zip':
            size_gb = entry.stat().st_size / 1e9
            print(f'  {entry.name:<55s} {size_gb:>6.2f} GB')
        elif entry.is_dir():
            print(f'  {entry.name}/  (dir)')
else:
    raise RuntimeError(f'{DATASETS_DRIVE} not found — is Drive mounted?')
print()

MULTISECTOR_RE = re.compile(r'^dataset_([a-z-]+)_(energy|water|transport|telecom)_v1_1k$')

def discover_multisector_sources():
    sources, seen = [], set()
    for entry in sorted(drive_path.iterdir()):
        stem = entry.stem if entry.suffix == '.zip' else entry.name
        m = MULTISECTOR_RE.match(stem)
        if not m:
            continue
        region, sector = m.group(1), m.group(2)
        if region not in REGIONS:
            continue
        key = (region, sector)
        if key in seen:
            continue
        seen.add(key)
        sources.append((region, sector, entry, entry.suffix == '.zip'))
    return sources


def materialize_source(region, sector, src_path, is_zip, force=False):
    folder_name = f'dataset_{region}_{sector}_v1_1k'
    target_dir  = Path(DATASETS_LOCAL) / folder_name
    if not force and (target_dir / 'manifest.json').exists():
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [DONE]   {region:<22s} {sector:<10s} already present ({n} tiles)')
        return target_dir

    if not is_zip:
        if not (src_path / 'manifest.json').exists():
            print(f'  [EMPTY]  {region:<22s} {sector:<10s} no manifest.json')
            return None
        t0 = time.time()
        print(f'  [COPY]   {region:<22s} {sector:<10s} {src_path.name} ...', end=' ', flush=True)
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.copytree(src_path, target_dir)
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'done in {time.time()-t0:.0f}s ({n} tiles)')
        return target_dir

    # zip — handle both wrapped and unwrapped layouts
    t0 = time.time()
    print(f'  [EXTRACT]{region:<22s} {sector:<10s} {src_path.name} ...', end=' ', flush=True)
    with zipfile.ZipFile(src_path) as zf:
        names = zf.namelist()
        wrapped_prefix = f'{folder_name}/'
        is_wrapped = any(n.startswith(wrapped_prefix) for n in names)
        if is_wrapped:
            zf.extractall(DATASETS_LOCAL)
        else:
            target_dir.mkdir(parents=True, exist_ok=True)
            zf.extractall(target_dir)
    n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
    print(f'done in {time.time()-t0:.0f}s ({n} tiles)')
    return target_dir


discovered = discover_multisector_sources()
print(f'Found {len(discovered)} multi-sector v1 sources.\n')

ready = []
print('Materializing:')
for region, sector, src_path, is_zip in discovered:
    local = materialize_source(region, sector, src_path, is_zip)
    if local is None:
        continue
    if (local / 'manifest.json').exists() and any((local / 'images').glob('*.npy')):
        ready.append((region, sector, local))

print(f'\nReady: {len(ready)} (region, sector) pairs')


Contents of /content/drive/MyDrive/infra_fm/datasets:
  dataset_africa_energy_v1_1k.zip                           0.04 GB
  dataset_africa_stac_v1.zip                                0.18 GB
  dataset_africa_telecom_v1_1k.zip                          0.00 GB
  dataset_africa_transport_v1_1k.zip                        0.05 GB
  dataset_africa_water_v1_1k.zip                            0.04 GB
  dataset_asia_energy_v1_1k.zip                             0.04 GB
  dataset_asia_stac_v1.zip                                  0.76 GB
  dataset_asia_telecom_v1_1k.zip                            0.00 GB
  dataset_asia_transport_v1_1k.zip                          0.04 GB
  dataset_asia_water_v1_1k.zip                              0.05 GB
  dataset_australia-oceania_energy_v1_1k.zip                0.05 GB
  dataset_australia-oceania_stac_v1.zip                     0.11 GB
  dataset_australia-oceania_telecom_v1_1k (1).zip           0.00 GB
  dataset_australia-oceania_telecom_v1_1k.zip               0.

In [9]:
import json
import numpy as np
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from collections import Counter
import random


@dataclass
class _Record:
    path: 'Path'
    asset_id: str
    asset_type: str


def percentile_normalize(arr: np.ndarray, lo: float = PERC_LO, hi: float = PERC_HI) -> np.ndarray:
    """Clip to [lo, hi] percentile across the full array and rescale to [0, 1].
    Matches the helper used in the SatlasPretrain S2 dataset path so the
    optical comparison is apples-to-apples."""
    img = arr.astype(np.float32)
    low  = np.percentile(img, lo)
    high = np.percentile(img, hi)
    if high <= low:
        return np.clip(img / 255.0, 0.0, 1.0)
    return np.clip((img - low) / (high - low), 0.0, 1.0)


class PrithviDataset(Dataset):
    """Loads a single dataset_<region>_<sector>_v1_1k/ folder for Prithvi.

    Per-sample output shape: (6, T=1, H, W).
    The T=1 dimension is inserted here so the default collate stacks to
    (B, 6, 1, H, W) — Prithvi's 3D patch embedding expects (B, C, T, H, W).
    """
    def __init__(self, dataset_root, band_indices=PRITHVI_BAND_INDICES,
                 allowed_asset_types=tuple(ASSET_TYPE_MAP.keys())):
        self.dataset_root = Path(dataset_root)
        self.band_indices = list(band_indices)
        self.allowed = set(allowed_asset_types)

        manifest_path = self.dataset_root / 'manifest.json'
        if not manifest_path.exists():
            raise FileNotFoundError(f'missing manifest.json: {manifest_path}')
        with manifest_path.open() as f:
            manifest = json.load(f)

        records_in = manifest.get('records', [])
        images_dir = self.dataset_root / 'images'
        records, dropped = [], Counter()
        for r in records_in:
            at = r.get('asset_type')
            if not at:
                dropped['no_label'] += 1; continue
            if at not in self.allowed:
                dropped['filtered_type'] += 1; continue
            img_file = r.get('image_file')
            if not img_file:
                dropped['no_image_file'] += 1; continue
            p = images_dir / img_file
            if not p.exists():
                dropped['missing_npy'] += 1; continue
            try:
                arr = np.load(p, mmap_mode='r')
                if arr.shape[0] < max(self.band_indices) + 1:
                    dropped['too_few_bands'] += 1; continue
            except Exception:
                dropped['load_error'] += 1; continue
            records.append(_Record(path=p, asset_id=r.get('asset_id', p.stem), asset_type=at))
        if dropped:
            reasons = ', '.join(f'{k}={v}' for k, v in dropped.items())
            print(f'  PrithviDataset({self.dataset_root.name}): dropped '
                  f'{sum(dropped.values())} records ({reasons})')
        if not records:
            raise RuntimeError(f'no usable records in {self.dataset_root}')
        self.records = records

    def __len__(self):
        return len(self.records)

    def _load_image(self, path):
        arr = np.load(path)                           # (C, H, W) float32
        arr = arr[self.band_indices, :, :]            # -> (6, H, W) in Prithvi band order
        arr = percentile_normalize(arr)               # -> [0, 1]
        return arr.astype(np.float32)

    def __getitem__(self, idx):
        r = self.records[idx]
        img = self._load_image(r.path)                # (6, H, W)
        t = torch.from_numpy(img).unsqueeze(1)        # (6, 1, H, W)  — T=1 dim
        return {
            'image':      t,
            'asset_id':   r.asset_id,
            'asset_type': r.asset_type,
        }


class MultiSectorLabelWrapper(Dataset):
    """Adds integer label + resize + region/sector tagging."""
    def __init__(self, base, region, sector, input_size=IMAGE_SIZE):
        self.base = base
        self.region = region
        self.sector = sector
        self.input_size = input_size
        self.valid_indices, self.labels = [], []
        for i, r in enumerate(base.records):
            mapped = ASSET_TYPE_MAP.get(r.asset_type)
            if mapped is None:
                continue
            self.valid_indices.append(i)
            self.labels.append(CLASS_TO_IDX[mapped])

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        sample = self.base[self.valid_indices[idx]]
        img = sample['image']                          # (6, 1, H, W)
        # Resize spatial dims to IMAGE_SIZE. F.interpolate on a 4D tensor
        # treats the second dim as channels — but we have (C, T, H, W). To
        # resize H, W while keeping the T dim, fold T into the batch slot:
        C, T, H, W = img.shape
        img = img.reshape(C * T, 1, H, W)              # (C*T, 1, H, W) so interp resizes H, W
        img = F.interpolate(img, size=(self.input_size, self.input_size),
                            mode='bilinear', align_corners=False)
        img = img.reshape(C, T, self.input_size, self.input_size)   # back to (C, T, H', W')
        return {
            'image':    img,
            'label':    self.labels[idx],
            'asset_id': sample['asset_id'],
            'region':   self.region,
            'sector':   self.sector,
        }


class SubsetView(Dataset):
    def __init__(self, base, indices):
        self.base, self.indices = base, indices
        self.region, self.sector = base.region, base.sector
    def __len__(self): return len(self.indices)
    def __getitem__(self, i): return self.base[self.indices[i]]


def stratified_split(dataset, train_frac=0.7, val_frac=0.15, seed=SEED):
    by_class = {}
    for i, label in enumerate(dataset.labels):
        by_class.setdefault(label, []).append(i)
    rng = random.Random(seed)
    train, val, test = [], [], []
    for cls in sorted(by_class):
        idxs = by_class[cls].copy()
        rng.shuffle(idxs)
        n = len(idxs); n_tr = int(n * train_frac); n_va = int(n * val_frac)
        train.extend(idxs[:n_tr])
        val.extend(idxs[n_tr:n_tr + n_va])
        test.extend(idxs[n_tr + n_va:])
    return train, val, test


print('PrithviDataset + wrappers + stratified split helpers ready.')


PrithviDataset + wrappers + stratified split helpers ready.


In [10]:
source_datasets = {}
for region, sector, local in ready:
    try:
        base = PrithviDataset(local)
    except Exception as e:
        print(f'  {region:<22s} {sector:<10s} FAILED: {e}')
        continue
    ds = MultiSectorLabelWrapper(base, region=region, sector=sector)
    if len(ds) == 0:
        print(f'  {region:<22s} {sector:<10s} 0 valid samples (skipped)')
        continue
    source_datasets[(region, sector)] = ds

splits = {}
print(f'\n{"region":<22s} {"sector":<10s} {"train":>8s} {"val":>6s} {"test":>6s}')
print('-' * 60)
for key, ds in source_datasets.items():
    region, sector = key
    tr, va, te = stratified_split(ds)
    splits[key] = {
        'train': SubsetView(ds, tr),
        'val':   SubsetView(ds, va),
        'test':  SubsetView(ds, te),
    }
    print(f'{region:<22s} {sector:<10s} {len(tr):>8d} {len(va):>6d} {len(te):>6d}')

train_global = ConcatDataset([s['train'] for s in splits.values()])
val_global   = ConcatDataset([s['val']   for s in splits.values()])
test_global  = ConcatDataset([s['test']  for s in splits.values()])
print(f'\nGlobal: train={len(train_global)}  val={len(val_global)}  test={len(test_global)}')

# Combined class distribution.
print('\n=== Class distribution across all sources ===')
all_counts = Counter()
for ds in source_datasets.values():
    all_counts.update(ds.labels)
total = sum(all_counts.values())
for c, name in enumerate(CLASS_NAMES):
    n = all_counts.get(c, 0)
    pct = (100.0 * n / total) if total else 0.0
    print(f'  [{c:>2d}] {name:<34s} n={n:>6d}  ({pct:>5.2f}%)')
print(f'        {"TOTAL":<34s} n={total:>6d}')



region                 sector        train    val   test
------------------------------------------------------------
africa                 energy          661    141    147
africa                 telecom           0      0      1
africa                 transport       699    149    152
africa                 water           623    132    137
asia                   energy          619    131    139
asia                   telecom          19      4      5
asia                   transport       623    132    136
asia                   water           670    141    147
australia-oceania      energy          699    147    156
australia-oceania      telecom           8      1      3
australia-oceania      transport       698    149    153
australia-oceania      water           698    149    153
central-america        energy          696    147    156
central-america        telecom           0      0      1
central-america        transport       395     83     88
central-america        wat

In [19]:
import torch.nn as nn


class PrithviBackbone(nn.Module):
    """Prithvi-EO-2.0-300M-TL frozen backbone, mean-pooled patch features.

    Loader strategy:
      *** TerraTorch path DISABLED *** — a known Colab scipy/dask/rapids 
      dependency conflict (TypeError: xp_capabilities() got an unexpected 
      keyword argument 'out_of_scope') prevents terratorch from importing 
      in the current environment. We go straight to the transformers 
      fallback, which works because Prithvi's HF repo ships custom 
      modeling code (modeling_prithvi.py) via trust_remote_code=True.

    Feature extraction:
      Prithvi's encoder returns patch-token sequences `(B, N, D)` where N
      includes the spatial patches across all timesteps (no CLS token in the
      standard ViT-MAE setup used by Prithvi). We mean-pool over the token
      dim to get `(B, D)` — preferred over CLS-token even when CLS is
      present, per the spec note (more stable for classification). The
      forward is defensive against a handful of possible output shapes that
      different terratorch / transformers versions return.
    """
    NAME = 'prithvi_eo_v2_300m_tl'
    HF_REPO = 'ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL'
    EXPECTED_FEATURE_DIM = 1024   # 300M variant uses ViT-L hidden dim

    def __init__(self, freeze=True):
        super().__init__()
        self._loader_used = None
        self.backbone = self._load_backbone()
        # Probe a forward pass to infer feature_dim. We require the user has
        # already constructed train_global so we can grab a sample.
        self.feature_dim = self._infer_feature_dim()
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def _load_backbone(self):
        # ---- TerraTorch path SKIPPED due to Colab dependency conflict ----
        # (Original try-block kept commented for reference; restore once
        # the scipy/dask/rapids issue is resolved upstream.)
        #
        # try:
        #     from terratorch.registry import BACKBONE_REGISTRY
        #     for name in ('prithvi_eo_v2_300_tl',
        #                  'prithvi_eo_v2_300m_tl',
        #                  'prithvi_v2_300_tl',
        #                  'prithvi_eo_300_tl'):
        #         try:
        #             bb = BACKBONE_REGISTRY.build(name, pretrained=True)
        #             print(f'  Loaded via terratorch.registry: {name!r}')
        #             self._loader_used = f'terratorch:{name}'
        #             return bb
        #         except Exception as e:
        #             last_err = e
        #             continue
        # except ImportError:
        #     pass
        print('  TerraTorch path skipped (known dependency conflict); '
              'using transformers fallback.')

        # ---- transformers AutoModel with trust_remote_code ----
        try:
            from transformers import AutoModel
            bb = AutoModel.from_pretrained(self.HF_REPO, trust_remote_code=True, num_labels=0)
            print(f'  Loaded via transformers.AutoModel: {self.HF_REPO}')
            self._loader_used = 'transformers'
            return bb
        except Exception as e:
            raise RuntimeError(
                f'Could not load Prithvi-EO-2.0 via transformers fallback. '
                f'Error: {e.__class__.__name__}: {e}. '
                f'Verify HuggingFace authentication (huggingface-cli login or '
                f'HF_TOKEN env var) and that transformers is installed.'
            )

    def _probe_forward(self, dummy):
        """Try the common calling conventions for the loaded backbone and
        return whatever the encoder produces. Used both for dim inference
        and inside `forward`."""
        # Prithvi-EO-2.0-TL accepts optional temporal_coords and location_coords
        # tensors. For our single-timestamp, no-metadata case we pass None.
        attempts = [
            lambda: self.backbone(dummy),
            lambda: self.backbone(dummy, temporal_coords=None, location_coords=None),
            lambda: self.backbone(pixel_values=dummy),
            lambda: self.backbone(pixel_values=dummy, temporal_coords=None, location_coords=None),
        ]
        last_err = None
        for attempt in attempts:
            try:
                return attempt()
            except (TypeError, RuntimeError) as e:
                last_err = e
        raise RuntimeError(f'No Prithvi forward signature worked. Last error: {last_err}')

    @staticmethod
    def _extract_features(out):
        """Reduce whatever the encoder returned to a (B, D) feature tensor."""
        # Case 1: tuple/list of tensors -> take the LAST (final-stage features)
        if isinstance(out, (tuple, list)):
            out = out[-1]
        # Case 2: a HuggingFace ModelOutput -> prefer last_hidden_state
        if hasattr(out, 'last_hidden_state'):
            out = out.last_hidden_state
        # Now `out` should be a tensor. Reduce to (B, D).
        if out.dim() == 3:                  # (B, N, D) — mean-pool tokens
            return out.mean(dim=1)
        if out.dim() == 4:                  # (B, D, H, W) — global avg pool
            return out.mean(dim=[2, 3])
        if out.dim() == 5:                  # (B, D, T, H, W)
            return out.mean(dim=[2, 3, 4])
        raise RuntimeError(f'Unexpected Prithvi output shape: {tuple(out.shape)}')

    def _infer_feature_dim(self):
        # Build a tiny dummy: (B=1, C=6, T=1, H=IMAGE_SIZE, W=IMAGE_SIZE)
        self.backbone.eval()
        device = next(self.backbone.parameters()).device
        dummy = torch.zeros(1, 6, 1, IMAGE_SIZE, IMAGE_SIZE, device=device)
        with torch.no_grad():
            out = self._probe_forward(dummy)
            feat = self._extract_features(out)
        d = feat.shape[-1]
        if d != self.EXPECTED_FEATURE_DIM:
            print(f'  WARNING: feature_dim={d} (expected {self.EXPECTED_FEATURE_DIM} '
                  f'for 300M variant). Using observed dim.')
        else:
            print(f'  feature_dim = {d} (matches expected for Prithvi-EO-2.0 300M)')
        return d

    def forward(self, x):
        # x: (B, 6, H, W) or (B, 6, T, H, W). Ensure 5D for Prithvi.
        if x.dim() == 4:
            x = x.unsqueeze(2)              # add T=1 -> (B, 6, 1, H, W)
        out = self._probe_forward(x)
        return self._extract_features(out)  # -> (B, D)


class InfraBenchClassifier(nn.Module):
    def __init__(self, backbone, num_classes, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(backbone.feature_dim, num_classes),
        )
    def forward(self, x):
        return self.head(self.backbone(x))


print('PrithviBackbone + InfraBenchClassifier defined.')

PrithviBackbone + InfraBenchClassifier defined.


In [20]:
from torch.optim import AdamW
from collections import defaultdict
from sklearn.metrics import f1_score, confusion_matrix
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


def _labels_from(dataset):
    if isinstance(dataset, ConcatDataset):
        for d in dataset.datasets:
            yield from _labels_from(d)
    elif isinstance(dataset, SubsetView):
        for i in dataset.indices:
            yield dataset.base.labels[i]
    else:
        yield from dataset.labels


def compute_class_weights(train_set, max_weight=10.0):
    counts = Counter(_labels_from(train_set))
    total = sum(counts.values())
    weights = torch.zeros(len(CLASS_NAMES))
    for c in range(len(CLASS_NAMES)):
        if counts.get(c, 0) > 0:
            w = total / (len(CLASS_NAMES) * counts[c])
            weights[c] = min(w, max_weight)
    return weights


def collate(batch):
    return {
        'image':  torch.stack([b['image'] for b in batch]),   # (B, 6, 1, H, W)
        'label':  torch.tensor([b['label'] for b in batch], dtype=torch.long),
        'region': [b['region'] for b in batch],
        'sector': [b['sector'] for b in batch],
    }


@torch.no_grad()
def evaluate(model, loader, return_breakdowns=False):
    model.eval()
    all_preds, all_labels, all_regions, all_sectors = [], [], [], []
    for batch in loader:
        logits = model(batch['image'].to(DEVICE, non_blocking=True))
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['label'].numpy().tolist())
        all_regions.extend(batch['region'])
        all_sectors.extend(batch['sector'])
    result = {
        'acc': float(np.mean(np.array(all_preds) == np.array(all_labels))),
        'macro_f1': float(f1_score(all_labels, all_preds, average='macro', zero_division=0.0)),
        'per_class_f1': f1_score(all_labels, all_preds, average=None,
                                 labels=list(range(len(CLASS_NAMES))),
                                 zero_division=0.0).tolist(),
        'confusion': confusion_matrix(all_labels, all_preds,
                                      labels=list(range(len(CLASS_NAMES)))).tolist(),
    }
    if return_breakdowns:
        def grouped_f1(group_vals):
            groups = defaultdict(lambda: {'preds': [], 'labels': []})
            for p, l, g in zip(all_preds, all_labels, group_vals):
                groups[g]['preds'].append(p)
                groups[g]['labels'].append(l)
            return {
                g: {
                    'n': len(d['labels']),
                    'macro_f1': float(f1_score(d['labels'], d['preds'], average='macro',
                                               zero_division=0.0)),
                    'acc': float(np.mean(np.array(d['preds']) == np.array(d['labels']))),
                }
                for g, d in groups.items()
            }
        result['per_region'] = grouped_f1(all_regions)
        result['per_sector'] = grouped_f1(all_sectors)
    return result


def train_one_run(backbone_factory, train_set, val_set, test_set, *,
                  freeze_backbone, num_epochs, batch_size, lr,
                  weight_decay=1e-4, num_workers=2, run_name='run'):
    backbone = backbone_factory(freeze=freeze_backbone)
    model = InfraBenchClassifier(backbone, num_classes=len(CLASS_NAMES)).to(DEVICE)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, collate_fn=collate, pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, collate_fn=collate, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, collate_fn=collate, pin_memory=True)

    weights = compute_class_weights(train_set).to(DEVICE)
    print('  Class weights (capped at 10.0):')
    for c, w in enumerate(weights.cpu().numpy()):
        print(f'    [{c:>2d}] {CLASS_NAMES[c]:<34s} {w:.4f}')
    criterion = nn.CrossEntropyLoss(weight=weights)

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable, lr=lr, weight_decay=weight_decay)

    ckpt_dir = Path(OUTPUT_DIR) / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt_path  = ckpt_dir / 'checkpoint_best.pt'
    final_ckpt_path = ckpt_dir / 'checkpoint_final.pt'

    history, best_val_f1, best_epoch = [], -1.0, -1
    for epoch in range(num_epochs):
        model.train()
        t0 = time.time()
        loss_sum, n = 0.0, 0
        for batch in train_loader:
            images = batch['image'].to(DEVICE, non_blocking=True)
            labels = batch['label'].to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * images.size(0)
            n += images.size(0)
        train_loss = loss_sum / max(n, 1)
        val = evaluate(model, val_loader)
        history.append({
            'epoch': epoch + 1, 'train_loss': train_loss,
            'val_acc': val['acc'], 'val_macro_f1': val['macro_f1'],
            'time_s': time.time() - t0,
        })

        marker = ''
        if val['macro_f1'] > best_val_f1:
            best_val_f1 = val['macro_f1']
            best_epoch  = epoch + 1
            marker = ' *'
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'val_macro_f1': val['macro_f1'],
                'history': history,
            }, best_ckpt_path)
        print(f'  ep {epoch+1:>3d}  loss={train_loss:.4f}  '
              f'val_acc={val["acc"]:.4f}  val_f1={val["macro_f1"]:.4f}{marker}')

    # Always save final-epoch state too.
    torch.save({
        'epoch': num_epochs,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'history': history,
        'best_val_f1': best_val_f1,
        'best_epoch':  best_epoch,
    }, final_ckpt_path)

    # ============== BEST_CKPT_BEFORE_TEST =================================
    # Restore best-val weights BEFORE running the held-out test pass. This
    # fixes the methodological gap in the earlier S1 run (test used final-
    # epoch weights, which can lag the best-val state by several epochs of
    # overfitting). Documented in the spec; deliberately built in here.
    # =====================================================================
    if best_ckpt_path.exists():
        ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f'\n  [BEST_CKPT_BEFORE_TEST] restored best-val state from '
              f'epoch {ckpt["epoch"]} (val_f1={ckpt["val_macro_f1"]:.4f}) '
              f'before test evaluation')
    else:
        print('\n  WARNING: no best-val checkpoint was saved (val_f1 never '
              'exceeded -1.0). Test will run on final-epoch state. '
              'Investigate before trusting these test numbers.')

    tail = [h['val_macro_f1'] for h in history[-min(5, len(history)):]]
    test = evaluate(model, test_loader, return_breakdowns=True)
    return {
        'run_name': run_name,
        'backbone': backbone.NAME,
        'condition': 'linear_probe' if freeze_backbone else 'fine_tune',
        'num_epochs': num_epochs,
        'best_val_f1': best_val_f1,
        'best_epoch':  best_epoch,
        'tail_mean_f1': float(np.mean(tail)),
        'tail_std_f1':  float(np.std(tail)),
        'history': history,
        'test': test,
        'tested_with': f'best-val checkpoint from epoch {best_epoch}'
                       if best_ckpt_path.exists() else 'final-epoch state (no best ckpt found)',
    }


print('Training infrastructure ready (with best-val checkpoint restore before test).')


Device: cuda
Training infrastructure ready (with best-val checkpoint restore before test).


In [21]:
# ============================================================================
# SMOKE CHECK — load backbone, build one batch, run forward + one train step.
# Default SMOKE_ONLY=True so the invocation cell below exits without training.
# ============================================================================

SMOKE_ONLY = False

def build_prithvi_backbone(freeze):
    return PrithviBackbone(freeze=freeze)

print('Loading Prithvi-EO-2.0-300M-TL backbone (frozen)...')
backbone = build_prithvi_backbone(freeze=True)
model = InfraBenchClassifier(backbone, num_classes=len(CLASS_NAMES)).to(DEVICE)

n_total     = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Loader used:      {backbone._loader_used}')
print(f'  Feature dim:      {backbone.feature_dim}')
print(f'  Total params:     {n_total:>13,}')
print(f'  Trainable params: {n_trainable:>13,}  (linear probe: should be << total)')

print('\nPulling one batch from train_global...')
smoke_loader = DataLoader(train_global, batch_size=4, shuffle=False,
                          num_workers=0, collate_fn=collate)
batch = next(iter(smoke_loader))
img = batch['image']
print(f'  Batch image shape:  {tuple(img.shape)}  (expect [4, 6, 1, {IMAGE_SIZE}, {IMAGE_SIZE}])')
print(f'  Batch image dtype:  {img.dtype}')
print(f'  Batch image range:  [{img.min().item():.4f}, {img.max().item():.4f}]  (expect ~[0, 1])')
print(f'  Batch image mean:   {img.mean().item():.4f}')
print(f'  Batch labels:       {batch["label"].tolist()}')
print(f'  Batch regions:      {batch["region"]}')
print(f'  Batch sectors:      {batch["sector"]}')

# ---- Forward pass ----
print('\nForward pass (frozen backbone, eval mode)...')
model.eval()
with torch.no_grad():
    feats = backbone(img.to(DEVICE))
    logits = model(img.to(DEVICE))
print(f'  Backbone features shape: {tuple(feats.shape)}  (expect [4, {backbone.feature_dim}])')
print(f'  Classifier logits shape: {tuple(logits.shape)}  (expect [4, {len(CLASS_NAMES)}])')
print(f'  Logits range:            [{logits.min().item():.4f}, {logits.max().item():.4f}]')
print(f'  Predicted classes:       {logits.argmax(dim=1).cpu().tolist()}')

# NaN/Inf guard
finite_logits = torch.isfinite(logits).all().item()
finite_feats  = torch.isfinite(feats).all().item()
print(f'  All finite (features):   {finite_feats}')
print(f'  All finite (logits):     {finite_logits}')
assert finite_feats and finite_logits, 'NaN/Inf detected in forward pass — aborting smoke check'

# ---- One train step ----
print('\nOne training step (verifies head trains end-to-end)...')
model.train()
weights = compute_class_weights(train_global).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable, lr=LP_LR, weight_decay=1e-4)
optimizer.zero_grad()
loss = criterion(model(img.to(DEVICE)), batch['label'].to(DEVICE))
loss.backward()
# Confirm head got nonzero grads
head_grads = [p.grad for p in model.head.parameters() if p.grad is not None]
assert head_grads and any(g.abs().sum().item() > 0 for g in head_grads), \
    'head received zero gradients — something is wrong with the loss/forward path'
optimizer.step()
print(f'  Train step loss:         {loss.item():.4f}  (finite={torch.isfinite(loss).item()})')
print(f'  Head grad sum (abs):     {sum(g.abs().sum().item() for g in head_grads):.4f}')

print('\nSmoke check PASSED — backbone loads, forward + backward + step all work.')
print(f'\nSMOKE_ONLY = {SMOKE_ONLY}  — the invocation cell below will '
      f'{"NOT train" if SMOKE_ONLY else "TRAIN"}.')


Loading Prithvi-EO-2.0-300M-TL backbone (frozen)...
  TerraTorch path skipped (known dependency conflict); using transformers fallback.


RuntimeError: Could not load Prithvi-EO-2.0 via transformers fallback. Error: TypeError: 'NoneType' object cannot be interpreted as an integer. Verify HuggingFace authentication (huggingface-cli login or HF_TOKEN env var) and that transformers is installed.

In [ ]:
# ============================================================================
# LINEAR PROBE INVOCATION — gated by SMOKE_ONLY from the cell above.
# ============================================================================
import json as _json

if SMOKE_ONLY:
    print('SMOKE_ONLY=True — skipping training. '
          'Set SMOKE_ONLY=False in the smoke-check cell above, re-run that '
          'cell and then this one to start the overnight run.')
else:
    print('=' * 70)
    print('Prithvi-EO-2.0-300M-TL — Linear Probe (frozen backbone)')
    print(f'  REGIONS = {len(REGIONS)}, classes = {len(CLASS_NAMES)}, '
          f'epochs = {LP_EPOCHS}, batch = {LP_BATCH}, lr = {LP_LR}')
    print('=' * 70)

    results = {}
    results['linear_probe'] = train_one_run(
        backbone_factory=build_prithvi_backbone,
        train_set=train_global, val_set=val_global, test_set=test_global,
        freeze_backbone=True,
        num_epochs=LP_EPOCHS,
        batch_size=LP_BATCH,
        lr=LP_LR,
        run_name='prithvi_eo_v2_300m_tl_v1_linear_probe',
    )

    out_path = Path(OUTPUT_DIR) / 'prithvi_eo_v2_300m_tl_v1_linear_probe_results.json'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, 'w') as f:
        _json.dump(results, f, indent=2)
    print(f'\nResults saved to: {out_path}')

    # ---- Summary ----
    lp = results['linear_probe']
    print(f'\n=== Prithvi-EO-2.0 Linear Probe Summary ===')
    print(f'Tested with:     {lp["tested_with"]}')
    print(f'Best val F1:     {lp["best_val_f1"]:.4f}  (epoch {lp["best_epoch"]})')
    print(f'Tail mean F1:    {lp["tail_mean_f1"]:.4f} +/- {lp["tail_std_f1"]:.4f}')
    print(f'Test macro F1:   {lp["test"]["macro_f1"]:.4f}')
    print(f'Test accuracy:   {lp["test"]["acc"]:.4f}')

    print(f'\nPer-class F1 (test):')
    for c, name in enumerate(CLASS_NAMES):
        print(f'  [{c:>2d}] {name:<34s} {lp["test"]["per_class_f1"][c]:.4f}')

    print(f'\nPer-sector F1 (test):')
    for sector, stats in sorted(lp['test']['per_sector'].items()):
        print(f'  {sector:<10s} n={stats["n"]:>5d}  F1={stats["macro_f1"]:.4f}  acc={stats["acc"]:.4f}')

    print(f'\nPer-region F1 (test):')
    for region, stats in sorted(lp['test']['per_region'].items()):
        print(f'  {region:<22s} n={stats["n"]:>5d}  F1={stats["macro_f1"]:.4f}  acc={stats["acc"]:.4f}')


Prithvi-EO-2.0-300M-TL — Linear Probe (frozen backbone)
  REGIONS = 7, classes = 13, epochs = 25, batch = 16, lr = 0.001
  Loaded via terratorch.registry: 'prithvi_eo_v2_300_tl'
  feature_dim = 1024 (matches expected for Prithvi-EO-2.0 300M)
  Class weights (capped at 10.0):
    [ 0] energy.transmission.substation     2.1460
    [ 1] energy.distribution.substation     1.4823
    [ 2] energy.distribution.other          0.4412
    [ 3] energy.generation.power_plant      2.6626
    [ 4] energy.generation.solar_farm       2.2979
    [ 5] energy.generation.wind_farm        10.0000
    [ 6] water.wastewater.plant             1.1097
    [ 7] water.treatment.plant              1.6155
    [ 8] water.storage_tank                 0.3604
    [ 9] transport.airport                  1.6554
    [10] transport.train_station            0.2852
    [11] transport.port_terminal            10.0000
    [12] telecom.data_center                2.7349
  ep   1  loss=2.2376  val_acc=0.3033  val_f1=0.1965 *
  ep